In [13]:
# Created on Oct 10, 2025
# Created by: Afeena
# Purpose: This file answers the business questions for the user slice

import pandas as pd
import numpy as np
import os as os
# import sys
import statsmodels.formula.api as smf
from itertools import product


# File Paths
EnvSeveritySlicePath = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\04Regression\\EnvSeveritySlice.csv"
destinationFolder = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\04Regression"

# Read the CSV file
dfEnv = pd.read_csv(EnvSeveritySlicePath, low_memory=False)

# Make column for SevereInjury:Fatal/Major = 1, Minor/Minimal = 0
dfEnv['SevereInjury'] = np.where(dfEnv['SeverityOfInjury'].isin(['Fatal', 'Major']), 1, 0)

# print(df['SeverityOfInjury'].value_counts())
# print(df['SevereInjury'].value_counts())

# identify dummy variable columns
envLightCondCols = [c for c in dfEnv.columns if c.startswith('EnvLightingCondition_')]
envRoadSurfCols = [c for c in dfEnv.columns if c.startswith('EnvRoadSurfaceCondition_')]

# baseline have already been dropped in Wrangler.ipynb

print ("environment light conditions dummies:", envLightCondCols[:5],"...total:", len(envLightCondCols))
print ("road surface conditions dummies:", envRoadSurfCols[:5],"...total:", len(envRoadSurfCols))

# ModelA: Which user involvement types are at the most risk of severe injuries compared to drivers?
formulaA = 'SevereInjury ~ ' + ' + '.join(envLightCondCols)
modelA = smf.logit(formula=formulaA, data=dfEnv).fit()
print("\n Model A: Lighting Conditions vs Severe Injuries \n")
print(modelA.summary())

# Save model summary to csv file
with open(os.path.join(destinationFolder, "EnvModelA_Stats.csv"), 'w') as f:
    f.write(str(modelA.summary()))

# ModelB: How do the age ranges of involved persons influence the probability of severe injuries?
formulaB = 'SevereInjury ~ ' + ' + '.join(envRoadSurfCols)
modelB = smf.logit(formula=formulaB, data=dfEnv).fit()
print("\n Model B: Road Surface Conditions vs Severe Injuries \n")
print(modelB.summary())

# Save model summary to csv file
with open(os.path.join(destinationFolder, "EnvModelB_Stats.csv"), 'w') as f:
    f.write(str(modelB.summary()))

# Function for Odds Ratios and Confidence Intervals
# the Odd Ratios and Summary function above will give the same results.
def odds_ratios_and_ci(whichModel):
    params = whichModel.params
    conf = whichModel.conf_int()
    out = pd.DataFrame({
        # Odds Ratios
        "OR": np.exp(params),
        # Confidence Intervals
        "2.5%": np.exp(conf[0]),
        "97.5%": np.exp(conf[1]),
        # p-values
        "pval": whichModel.pvalues
    })
    return out.sort_values(by="OR", ascending=False)

# Function Predicted probabilities for each involvement dummy (set one = 1, others = 0; hold rest at means)

# def predicted_probabilities(whichModel, dummy_cols):
#     # Get the mean values of the continuous variables
#     means = dfUser.mean()
#     # Create a DataFrame to hold the predicted probabilities
#     preds = pd.DataFrame(index=dummy_cols)
#     for col in dummy_cols:
#         # Set the current dummy variable to 1 and others to 0
#         X = pd.DataFrame(0, index=np.arange(1), columns=dummy_cols)
#         X[col] = 1
#         # Add the mean values of the continuous variables
#         for var in means.index:
#             if var not in dummy_cols:
#                 X[var] = means[var]
#         # Get the predicted probabilities
#         preds[col] = whichModel.predict(X)
#     return preds

# Predicted probability for each involvement dummy (set one=1, others=0; all other dummies=0)
def group_pred_by_singleton(whichModel, whichColumns, group_name="Group"):
    # create baseline scenario - set all dummy variables to 0 (reference categories). I will turn on what I want to test
    base = {col: 0 for col in whichModel.model.exog_names if col not in ["Intercept"]}
    # list to store prediction results
    results = []
    for g in whichColumns:
        # creates a copy to avoid modifying the original baseline
        scenario = base.copy()
        # Set only the current group dummy to 1, others stay 0, allowing me to loop through each dummy variable set it to 1 and then leave the others at 0
        scenario[g] = 1
        prob = whichModel.predict(pd.DataFrame([scenario]))[0]
        results.append({group_name: g, "PredProb_Severe": prob})
    return pd.DataFrame(results).sort_values("PredProb_Severe", ascending=False)

#Print and save Predicted probabilities for Models A and Model B
print("\n=== Predicted probability of severe injury by Environment Lighting condition ===")
predProb_Lighting = group_pred_by_singleton(modelA, envLightCondCols, group_name="EnvironmentLighting")
display(predProb_Lighting)
predProb_Lighting.to_csv(os.path.join(destinationFolder, "EnvModelA_Stats.csv"),  mode='a', header=True,index=False)

print("\n=== Predicted probability of severe injury by Road Surface Condition ===")
predProb_RoadSurface = group_pred_by_singleton(modelB, envRoadSurfCols, group_name="RoadSurface")
display(predProb_RoadSurface)
predProb_RoadSurface.to_csv(os.path.join(destinationFolder, "EnvModelB_Stats.csv"), mode='a', header=True, index=False)

# Print and save outputs of odds ratios
modelA_or_ci = odds_ratios_and_ci(modelA)
print("\n=== Odds Ratios for Model A (environment lighting condition) ===")
display(modelA_or_ci)
modelA_or_ci.to_csv(os.path.join(destinationFolder, "EnvModelA_Stats.csv"), mode='a', header=True, index=False)

modelB_or_ci = odds_ratios_and_ci(modelB)
print("\n=== Odds Ratios for Model B (Road Surface Condition) ===")
display(modelB_or_ci)
modelB_or_ci.to_csv(os.path.join(destinationFolder, "EnvModelB_Stats.csv"), mode='a', header=True, index=False)

# wet-dark predicted probability
# Fit logistic regression model

# formulaC = 'SevereInjury ~ ' + ' + '.join(envLightCondCols)
# modelC = smf.logit(formula=formulaC, data=dfEnv).fit()
# print("\n Model C: Wet-DDark vs Severe Injuries \n")
# print(modelC.summary())

formulaC="""
    SevereInjury ~ EnvLightingCondition_Dark 
                   + EnvLightingCondition_Dark__artificial
                   + EnvLightingCondition_Dusk
                   + EnvLightingCondition_Dusk__artificial
                   + EnvLightingCondition_Dawn
                   + EnvLightingCondition_Dawn__artificial
                   + EnvRoadSurfaceCondition_Wet
                   + EnvRoadSurfaceCondition_Ice
                   + EnvRoadSurfaceCondition_Packed_Snow
                   + EnvRoadSurfaceCondition_Slush
                   + EnvRoadSurfaceCondition_Loose_Snow
                   + EnvRoadSurfaceCondition_Loose_Sand_or_Gravel
                   + EnvRoadSurfaceCondition_Spilled_liquid
                   + EnvRoadSurfaceCondition_Other
    """
modelC = smf.logit(
    formula=formulaC,
    data=dfEnv
).fit()

# Create input for wet-dark conditions (baseline: Daylight, Dry)
wet_dark = pd.DataFrame({
    "EnvLightingCondition_Dark": [1],
    "EnvLightingCondition_Dark__artificial": [0],
    "EnvLightingCondition_Dawn": [0],
    "EnvLightingCondition_Dawn__artificial": [0],
    "EnvLightingCondition_Daylight__artificial": [0],
    "EnvLightingCondition_Dusk": [0],
    "EnvLightingCondition_Dusk__artificial": [0],
    "EnvLightingCondition_Other": [0],
    "EnvRoadSurfaceCondition_Ice": [0],
    "EnvRoadSurfaceCondition_Loose_Sand_or_Gravel": [0],
    "EnvRoadSurfaceCondition_Loose_Snow": [0],
    "EnvRoadSurfaceCondition_Other": [0],
    "EnvRoadSurfaceCondition_Packed_Snow": [0],
    "EnvRoadSurfaceCondition_Slush": [0],
    "EnvRoadSurfaceCondition_Spilled_liquid": [0],
    "EnvRoadSurfaceCondition_Wet": [1]
})

# Predict probability for Wet–Dark
pred_prob_wet_dark = modelC.predict(wet_dark)
print("\nPredicted Probability of Severe Collision (Wet–Dark):", float(pred_prob_wet_dark))
pred_prob_wet_dark.to_csv(os.path.join(destinationFolder, "EnvModelC_Stats.csv"),  mode='a', header=True,index=False)

# Find the deadliest coobminations of lighting and road surface conditions

# #  Generate all combinations of lighting × road surface conditions
# lightingCols = [
#     "EnvLightingCondition_Dark",
#     "EnvLightingCondition_Dark__artificial",
#     "EnvLightingCondition_Dusk",
#     "EnvLightingCondition_Dusk__artificial",
#     "EnvLightingCondition_Dawn",
#     "EnvLightingCondition_Dawn__artificial"
# ]

# road_vars = [
#     "EnvRoadSurfaceCondition_Wet",
#     "EnvRoadSurfaceCondition_Ice",
#     "EnvRoadSurfaceCondition_Packed_Snow",
#     "EnvRoadSurfaceCondition_Slush",
#     "EnvRoadSurfaceCondition_Loose_Snow",
#     "EnvRoadSurfaceCondition_Loose_Sand_or_Gravel",
#     "EnvRoadSurfaceCondition_Spilled_liquid",
#     "EnvRoadSurfaceCondition_Other"
# ]

combinations = list(product(envLightCondCols, envRoadSurfCols))

# Create DataFrame for each combination and predict probabilities
results = []

for light, road in combinations:
    # Set all dummy vars = 0
    row = {v: 0 for v in envLightCondCols + envRoadSurfCols}
    
    # Turn on this combination
    row[light] = 1
    row[road] = 1
    
    # Predict probability
    prob = modelC.predict(pd.DataFrame([row]))[0]
    
    results.append({
        "LightingCondition": light,
        "RoadSurfaceCondition": road,
        "PredictedProbability": prob
    })

# Create a results DataFrame and find the max
pred_df = pd.DataFrame(results)
most_dangerous = pred_df.sort_values("PredictedProbability", ascending=False).head(1)
print("Most Dangerous Combination:\n", most_dangerous)
most_dangerous.to_csv(os.path.join(destinationFolder, "EnvModelC_Stats.csv"),  mode='a', header=True,index=False)



environment light conditions dummies: ['EnvLightingCondition_Dark', 'EnvLightingCondition_Dark__artificial', 'EnvLightingCondition_Dawn', 'EnvLightingCondition_Dawn__artificial', 'EnvLightingCondition_Daylight__artificial'] ...total: 8
road surface conditions dummies: ['EnvRoadSurfaceCondition_Ice', 'EnvRoadSurfaceCondition_Loose_Sand_or_Gravel', 'EnvRoadSurfaceCondition_Loose_Snow', 'EnvRoadSurfaceCondition_Other', 'EnvRoadSurfaceCondition_Packed_Snow'] ...total: 8
         Current function value: 0.574694
         Iterations: 35

 Model A: Lighting Conditions vs Severe Injuries 

                           Logit Regression Results                           
Dep. Variable:           SevereInjury   No. Observations:                10044
Model:                          Logit   Df Residuals:                    10035
Method:                           MLE   Df Model:                            8
Date:                Mon, 06 Oct 2025   Pseudo R-squ.:                0.001572
Time:           

c:\Users\afeen\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\afeen\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,EnvironmentLighting,PredProb_Severe
7,EnvLightingCondition_Other,0.999992
6,EnvLightingCondition_Dusk__artificial,0.846774
3,EnvLightingCondition_Dawn__artificial,0.803571
4,EnvLightingCondition_Daylight__artificial,0.802469
5,EnvLightingCondition_Dusk,0.763780
1,EnvLightingCondition_Dark__artificial,0.730409
0,EnvLightingCondition_Dark,0.719245
2,EnvLightingCondition_Dawn,0.709677



=== Predicted probability of severe injury by Road Surface Condition ===


,RoadSurface,PredProb_Severe
6,EnvRoadSurfaceCondition_Spilled_liquid,1.000000
3,EnvRoadSurfaceCondition_Other,0.849315
4,EnvRoadSurfaceCondition_Packed_Snow,0.782609
7,EnvRoadSurfaceCondition_Wet,0.754819
1,EnvRoadSurfaceCondition_Loose_Sand_or_Gravel,0.750000
5,EnvRoadSurfaceCondition_Slush,0.711538
2,EnvRoadSurfaceCondition_Loose_Snow,0.670213
0,EnvRoadSurfaceCondition_Ice,0.537037


...

c:\Users\afeen\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)


,OR,2.5%,97.5%,pval
EnvRoadSurfaceCondition_Spilled_liquid,9.996549e+07,0.000000,inf,0.999118
Intercept,2.775234e+00,2.641427,2.915818,0.000000
EnvRoadSurfaceCondition_Other,2.030951e+00,1.067552,3.863759,0.030837
EnvRoadSurfaceCondition_Packed_Snow,1.297188e+00,0.481021,3.498175,0.607202
EnvRoadSurfaceCondition_Wet,1.109321e+00,0.981663,1.253579,0.096259
EnvRoadSurfaceCondition_Loose_Sand_or_Gravel,1.080990e+00,0.218015,5.359893,0.924049
EnvRoadSurfaceCondition_Slush,8.888141e-01,0.486834,1.622711,0.701150
EnvRoadSurfaceCondition_Loose_Snow,7.322836e-01,0.475014,1.128890,0.158253
EnvRoadSurfaceCondition_Ice,4.179828e-01,0.244266,0.715243,0.001459


         Current function value: 0.573762
         Iterations: 35

Predicted Probability of Severe Collision (Wet–Dark): 0.7408447383608747


c:\Users\afeen\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\afeen\AppData\Local\Temp\ipykernel_27512\558883133.py:178: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  print("\nPredicted Probability of Severe Collision (Wet–Dark):", float(pred_prob_wet_dark))


Most Dangerous Combination:
                         LightingCondition  \
54  EnvLightingCondition_Dusk__artificial   

                      RoadSurfaceCondition  PredictedProbability  
54  EnvRoadSurfaceCondition_Spilled_liquid                   1.0  
